In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from huggingface_hub import login, whoami
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
    print(whoami())
except Exception as e:
    print("Vui lòng thiết lập HF_TOKEN trong Colab Secrets.")

{'type': 'user', 'id': '6a53fd1cbc579eda5e1846ef', 'name': 'ntntloan', 'fullname': 'Nguyễn Trương Ngọc Thảo Loan', 'email': 'ntntloan@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/5f8f1bc8ca982ea73cc0ccbce2c6658b.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'my-token', 'role': 'read', 'createdAt': '2026-07-17T01:07:28.867Z'}}}


In [ ]:
!pip install -q transformers torch sentencepiece accelerate bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import gc
import time
import unicodedata
import os
import json
import sys

import torch
from bert_score import BERTScorer
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList, set_seed

PRO_DIR = "/content/drive/MyDrive/gen-poem/"
if PRO_DIR not in sys.path:
    sys.path.append(PRO_DIR)
from lucbat_processor import StrictLucBatProcessor, load_syllables

In [5]:
MODEL_NAME = "Llama 3.2 3B"
MODEL_PATH = "/content/drive/MyDrive/gen-poem/models/llama32-lucbat"
DATASET_FILE = "/content/drive/MyDrive/gen-poem/data/test.txt"
SYLLABLE_FILE = "/content/drive/MyDrive/gen-poem/data/train.txt"
CHECKPOINT_DIR = "/content/drive/MyDrive/gen-poem/checkpoints"
OUTPUT_EVAL_JSON = os.path.join(CHECKPOINT_DIR, "eval_checkpoint.json")
SAVE_EVERY = 50

SAMPLE_RATIO = 0.1
SEED = 42
MAX_NEW_TOKENS = 25
BATCH_SIZE = 64
MODES = ("no_rules", "with_rules")

GEN_KWARGS = dict(do_sample=True, temperature=0.8, top_p=0.9)

TRAC_MARKS = {"\u0301", "\u0303", "\u0309", "\u0323"}
HUYEN = "\u0300"
TONE_MARKS = TRAC_MARKS | {HUYEN}
VOWELS = set("aeiouyăâêôơư")
CONSONANTS = ["ngh", "ch", "gh", "gi", "kh", "ng", "nh", "ph", "qu", "th", "tr",
              "b", "c", "d", "đ", "g", "h", "k", "l", "m", "n", "p", "q", "r",
              "s", "t", "v", "x"]

def _decompose(word):
    return unicodedata.normalize("NFD", word.lower())

def get_tone(word):
    d = _decompose(word)
    if not any(c in VOWELS for c in d if c not in TONE_MARKS):
        return None
    return "T" if any(c in TRAC_MARKS for c in d) else "B"

def get_tone_mark(word):
    return "huyen" if HUYEN in _decompose(word) else "ngang"

def remove_tones(word):
    stripped = "".join(c for c in _decompose(word) if c not in TONE_MARKS)
    return unicodedata.normalize("NFC", stripped)

def get_rhyme_part(word):
    w = remove_tones(word.strip())
    for c in CONSONANTS:
        if w.startswith(c):
            return w[len(c):]
    return w

def is_rhyme(w1, w2):
    if not w1 or not w2:
        return False
    return get_rhyme_part(w1) == get_rhyme_part(w2)

def evaluate_luc_bat(cau_luc, cau_bat):
    luc = cau_luc.strip().split()
    bat = cau_bat.strip().split()
    # L05
    score_len = 10.0 if len(bat) == 8 else 0.0
    # L06 đến L09
    score_tone = 0.0
    if len(bat) >= 8:
        step = 30.0 / 4.0
        for pos, target in ((2, "B"), (4, "T"), (6, "B"), (8, "B")):
            if get_tone(bat[pos - 1]) == target:
                score_tone += step
    # L10
    score_rhyme = 0.0
    if len(luc) >= 6 and len(bat) >= 6 and is_rhyme(luc[5], bat[5]):
        score_rhyme = 60.0
    # L13
    mark_ok = 0.0
    if len(bat) >= 8 and get_tone_mark(bat[5]) != get_tone_mark(bat[7]):
        mark_ok = 1.0
    # L15
    dup_rhyme = 0.0
    if len(luc) >= 6 and len(bat) >= 6 and luc[5].lower() == bat[5].lower():
        dup_rhyme = 1.0
    return dict(score=score_len + score_tone + score_rhyme,
                length=score_len, tone=score_tone, rhyme=score_rhyme,
                mark=mark_ok, dup=dup_rhyme)

def load_test_pairs(path, ratio):
    with open(path, encoding="utf-8") as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    pairs = []
    i = 0
    while i < len(lines) - 1:
        if len(lines[i].split()) == 6 and len(lines[i + 1].split()) == 8:
            pairs.append((lines[i], lines[i + 1]))
            i += 2
        else:
            i += 1
    if ratio < 1.0:
        pairs = pairs[:max(1, int(len(pairs) * ratio))]
    return pairs

def pick_dtype():
    if torch.cuda.is_available():
        return torch.float16
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.float16
    return torch.float32

def generate_cau_bat(model, tokenizer, cau_luc, processor, seed):
    set_seed(seed)
    prompt = cau_luc + "\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    processors = LogitsProcessorList()
    if processor is not None:
        processors.append(processor)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            logits_processor=processors,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
            **GEN_KWARGS,
        )
    new_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    for line in text.split("\n"):
        if line.strip():
            return " ".join(line.split())
    return ""

def phobert_f1(scorer, generated, reference):
    scores = []
    with torch.no_grad():
        for idx in range(0, len(generated), BATCH_SIZE):
            g = generated[idx: idx + BATCH_SIZE]
            r = reference[idx: idx + BATCH_SIZE]
            P, R, F1 = scorer.score(g, r)
            scores.extend(F1.cpu().tolist())
            del P, R, F1
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
    return (sum(scores) / len(scores) * 100) if scores else 0.0

def load_checkpoint():
    if not os.path.exists(OUTPUT_EVAL_JSON):
        return {}
    try:
        with open(OUTPUT_EVAL_JSON, "r", encoding="utf-8") as f:
            checkpoint = json.load(f)
        print("Đã phát hiện checkpoint.")
        return checkpoint
    except Exception as e:
        print(f"Không thể nạp checkpoint. Chạy lại từ đầu. Lỗi: {e}")
        return {}

def save_checkpoint(checkpoint_data):
    try:
        with open(OUTPUT_EVAL_JSON, "w", encoding="utf-8") as f:
            json.dump(checkpoint_data, f, ensure_ascii=False, indent=4)
        return True
    except Exception as e:
        print(f"Lỗi ghi checkpoint: {e}")
        return False

def main():

    # ========================================================
    # LOAD DANH SÁCH TIẾNG HỢP LỆ
    # ========================================================

    syllables = None
    if SYLLABLE_FILE:
        try:
            syllables = load_syllables(SYLLABLE_FILE)
            print("Danh sách tiếng hợp lệ: %d tiếng" % len(syllables))
        except OSError:
            print("Không đọc được %s nên bỏ qua luật L16" % SYLLABLE_FILE)

    # ========================================================
    # LOAD TEST DATA
    # ========================================================

    test_data = load_test_pairs(DATASET_FILE, SAMPLE_RATIO)
    print("Số cặp câu lục và câu bát: %d" % len(test_data))


    # ========================================================
    # LOAD MODEL
    # ========================================================

    print("Đang tải mô hình %s" % MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        clean_up_tokenization_spaces=False,
        trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        dtype=pick_dtype(),
        attn_implementation="sdpa",
        trust_remote_code=True,
    )
    model.eval()

    # ========================================================
    # KHỞI TẠO PHOBERT
    # ========================================================

    print("Đang khởi tạo BERTScorer")
    scorer = BERTScorer(
        model_type="vinai/phobert-base",
        num_layers=9,
        rescale_with_baseline=False
    )

    # ========================================================
    # LOAD CHECKPOINT
    # ========================================================

    checkpoint_data = load_checkpoint()
    results = {}

    # ========================================================
    # CHẠY TỪNG CHẾ ĐỘ
    # ========================================================

    for mode in MODES:
        apply_rules = (mode == "with_rules")
        label = ("CÓ TẬP LUẬT" if apply_rules else "KHÔNG CÓ TẬP LUẬT")
        print("\n" + "=" * 62)
        print("ĐÁNH GIÁ %s TRÊN %d MẪU" % (label, len(test_data)))
        print("=" * 62)

        # ====================================================
        # KHỞI TẠO PROCESSOR
        # ====================================================

        processor = None
        if apply_rules:
            processor = StrictLucBatProcessor(
                tokenizer,
                max_total_lines=2,
                syllables=syllables
            )

        # ====================================================
        # GIÁ TRỊ MẶC ĐỊNH
        # ====================================================

        totals = dict(score=0.0, length=0.0, tone=0.0, rhyme=0.0, mark=0.0, dup=0.0)
        generated, reference = [], []
        start_index = 0
        accumulated_time = 0.0    # Thời gian đã chạy trước khi bị ngắt

        # ====================================================
        # KHÔI PHỤC CHECKPOINT CỦA RIÊNG MODE HIỆN TẠI
        # ====================================================

        if mode in checkpoint_data:
            mode_checkpoint = checkpoint_data[mode]
            start_index = mode_checkpoint.get("next_index", 0)
            generated = mode_checkpoint.get("generated", [])
            reference = mode_checkpoint.get("reference", [])
            saved_totals = mode_checkpoint.get("totals", {})
            for key in totals:
                totals[key] = saved_totals.get(key, 0.0)
            accumulated_time = mode_checkpoint.get("accumulated_time", 0.0)

            # -----------------------------------------------
            # KIỂM TRA TÍNH NHẤT QUÁN CỦA CHECKPOINT
            # -----------------------------------------------

            start_index = min(start_index,len(test_data))
            generated = generated[:start_index]
            reference = reference[:start_index]
            print(f"[!] Khôi phục checkpoint cho mode: {mode}")
            print(f"[!] Tiếp tục từ mẫu: {start_index}/{len(test_data)}")
            print(f"[!] Đã có: {len(generated)} câu sinh")
            print(f"[!] Thời gian đã tích lũy: {accumulated_time:.1f} giây")
        else:
            print(f"[!] Không có checkpoint cho mode {mode}. Chạy từ đầu.")

        # ====================================================
        # NẾU MODE NÀY ĐÃ HOÀN THÀNH
        # ====================================================

        if start_index >= len(test_data):
            print(f"[!] Mode '{mode}' đã chạy xong từ checkpoint.")

        # ====================================================
        # BẮT ĐẦU TÍNH THỜI GIAN PHIÊN HIỆN TẠI
        # ====================================================

        session_start_time = time.time()

        # ====================================================
        # VÒNG LẶP SINH THƠ: range(start_index, ...) thay vì enumerate(test_data) để resume chính xác.
        # ====================================================

        for i in range(start_index, len(test_data)):
            cau_luc, cau_bat_ref = test_data[i]

            # ------------------------------------------------
            # SINH CÂU BÁT
            # ------------------------------------------------

            cau_bat = generate_cau_bat(model, tokenizer, cau_luc, processor, SEED + i)

            # ------------------------------------------------
            # CHẤM ĐIỂM
            # ------------------------------------------------

            row = evaluate_luc_bat(cau_luc, cau_bat)
            for key in totals:
                totals[key] += row[key]
            generated.append(cau_bat)
            reference.append(cau_bat_ref)

            # =================================================
            # AUTO SAVE CHECKPOINT
            # =================================================

            if ((i + 1) % SAVE_EVERY == 0 or (i + 1) == len(test_data)):
                print("Đã xử lý [%d/%d] mẫu" % (i + 1, len(test_data)))

                # Thời gian phiên chạy hiện tại
                current_session_time = time.time() - session_start_time

                # Tổng thời gian từ các phiên trước + phiên hiện tại
                total_elapsed_time = accumulated_time + current_session_time

                # --------------------------------------------
                # CẬP NHẬT CHECKPOINT CỦA MODE HIỆN TẠI
                # --------------------------------------------

                checkpoint_data[mode] = {
                    "next_index": i + 1,
                    "generated": generated,
                    "reference": reference,
                    "totals": totals,
                    "accumulated_time": total_elapsed_time,
                    "completed": (i + 1) == len(test_data),
                    "total_samples": len(test_data),
                    "model_name": MODEL_NAME,
                    "mode": mode
                }

                # --------------------------------------------
                # GHI CHECKPOINT XUỐNG DISK / DRIVE
                # --------------------------------------------

                if save_checkpoint(checkpoint_data):
                    print(f"Đã lưu checkpoint mode '{mode}' tại mẫu {i + 1}/{len(test_data)}")

        # ====================================================
        # TÍNH THỜI GIAN SINH THƠ
        # ====================================================

        current_session_time = time.time() - session_start_time
        gen_time = accumulated_time + current_session_time


        # ====================================================
        # TÍNH PHOBERT
        # ====================================================

        print("[!] Đang tính PhoBERTScore")
        phobert_start_time = time.time()
        f1 = phobert_f1(scorer, generated, reference)
        phobert_time = time.time() - phobert_start_time

        # ====================================================
        # LƯU KẾT QUẢ MODE
        # ====================================================

        n = len(test_data)
        results[mode] = dict(
            score=totals["score"] / n,
            length=totals["length"] / n / 10 * 100,
            tone=totals["tone"] / n / 30 * 100,
            rhyme=totals["rhyme"] / n / 60 * 100,
            mark=totals["mark"] / n * 100,
            dup=totals["dup"] / n * 100,
            phobert=f1,
            seconds=gen_time,
            phobert_seconds=phobert_time,
            samples=generated[:5]
        )

        # ====================================================
        # LƯU TRẠNG THÁI HOÀN CHỈNH SAU MODE
        # ====================================================

        checkpoint_data[mode] = {
            "next_index": n,
            "generated": generated,
            "reference": reference,
            "totals": totals,
            "accumulated_time": gen_time,
            "completed": True,
            "total_samples": n,
            "model_name": MODEL_NAME,
            "mode": mode,
            "phobert": f1,
            "phobert_time": phobert_time
        }

        save_checkpoint(checkpoint_data)
        print(f"[!] Hoàn thành mode '{mode}'")

    # ========================================================
    # BÁO CÁO CUỐI CÙNG
    # ========================================================

    print("\n" + "=" * 62)
    print("BÁO CÁO ĐÁNH GIÁ MÔ HÌNH %s" % MODEL_NAME.upper())
    print("=" * 62)
    print("Tổng số mẫu: %d" % len(test_data))
    header = ("%-18s %9s %10s %10s %9s %8s %9s" % ("Chế độ", "Độ dài", "Thanh điệu", "Gieo vần", "Điểm luật", "PhoBERT", "Thời gian"))
    print(header)
    print("-" * len(header))
    for mode in MODES:
        r = results[mode]
        label = ("Có tập luật" if mode == "with_rules" else "Không tập luật")
        print("%-18s %8.1f%% %9.1f%% %9.1f%% %9.1f %7.1f%% %8.1f phút" % (label, r["length"], r["tone"], r["rhyme"], r["score"], r["phobert"], r["seconds"]/60))

    # ========================================================
    # SO SÁNH 2 MODE
    # ========================================================

    if len(MODES) == 2:
        a = results[MODES[0]]
        b = results[MODES[1]]
        print("-" * len(header))
        print("%-18s %8.1f%% %9.1f%% %9.1f%% %9.1f %7.1f%%" % ("Chênh lệch", b["length"] - a["length"], b["tone"] - a["tone"], b["rhyme"] - a["rhyme"], b["score"] - a["score"], b["phobert"] - a["phobert"]))

    # ========================================================
    # CÁC CHỈ SỐ BỔ SUNG
    # ========================================================

    print("\nCác chỉ số bổ sung")
    for mode in MODES:
        r = results[mode]
        label = ("Có tập luật" if mode == "with_rules" else "Không tập luật")
        print("%-18s khác dấu thanh %5.1f%% | trùng tiếng vần %5.1f%%" % (label, r["mark"], r["dup"]))

    # ========================================================
    # IN MỘT SỐ KẾT QUẢ
    # ========================================================

    print("\nMột số câu bát sinh ra")
    for i in range(min(5, len(test_data))):
        print("  Câu lục: %s" % test_data[i][0])
        for mode in MODES:
            label = ("không luật" if mode == "no_rules" else "có luật")
            print("    %-12s %s" % (label, results[mode]["samples"][i]))

    # ========================================================
    # LƯU KẾT QUẢ CUỐI CÙNG
    # ========================================================

    checkpoint_data["final_results"] = results
    save_checkpoint(checkpoint_data)
    print("\n[!] Đã lưu toàn bộ checkpoint và kết quả cuối cùng!")
    print(f"[!] File: {OUTPUT_EVAL_JSON}")

if __name__ == "__main__":
    main()

Danh sách tiếng hợp lệ: 9575 tiếng
Số cặp câu lục và câu bát: 28118
Đang tải mô hình Llama 3.2 3B


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Đang khởi tạo BERTScorer


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            


ĐÃ PHÁT HIỆN CHECKPOINT

ĐÁNH GIÁ KHÔNG CÓ TẬP LUẬT TRÊN 28118 MẪU
[!] KHÔI PHỤC CHECKPOINT CHO MODE: no_rules
[!] Tiếp tục từ mẫu: 24300/28118
[!] Đã có: 24300 câu sinh
[!] Thời gian đã tích lũy: 27127.6 giây
Đã xử lý [24350/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24350/28118
Đã xử lý [24400/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24400/28118
Đã xử lý [24450/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24450/28118
Đã xử lý [24500/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24500/28118
Đã xử lý [24550/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24550/28118
Đã xử lý [24600/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24600/28118
Đã xử lý [24650/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24650/28118
Đã xử lý [24700/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24700/28118
Đã xử lý [24750/28118] mẫu
Đã lưu checkpoint mode 'no_rules' tại mẫu 24750/28118
Đã xử lý [24800/28118] mẫu
Đã lưu checkpoint mode 'no_rules'